In [2]:
"""
Модуль 9. NLP в финансовом анализе (Sentiment Analysis)

Задача: Анализ влияния новостного фона на динамику финансовых инструментов.

Методология:
1. Natural Language Processing: Использование предобученного FinBERT для классификации тональности текстов
2. Aggregation: Построение агрегированного ежедневного индекса сентимента
3. Cross-correlation: Исследование корреляции между индексом сентимента и характеристиками рынка
4. Statistical Testing: Оценка значимости обнаруженных взаимосвязей

Источник текстов: HuggingFace Datasets — "financial_phrasebank" https://huggingface.co/datasets/takala/financial_phrasebank/tree/main/data
Рыночные данные: yfinance — котировки индекса S&P 500 (^GSPC) или DJIA (^DJI)
"""


# ================================
# 1. ИМПОРТ БИБЛИОТЕК
# ================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, spearmanr
from scipy.signal import correlate
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import yfinance as yf

# Hugging Face
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from transformers import pipeline

# Для подавления предупреждений
import warnings
warnings.filterwarnings('ignore')

# ================================
# 2. ЗАГРУЗКА ДАННЫХ
# ================================
print("=" * 60)
print("ЗАГРУЗКА ДАННЫХ")
print("=" * 60)

# TODO 2.1: Загрузите датасет "financial_phrasebank" с HuggingFace.
# Подсказка: используйте load_dataset("financial_phrasebank", "sentences_allagree")
# Эта версия датасета содержит 4840 предложений из финансовых новостей с метками: 0=negative, 1=neutral, 2=positive
dataset = # <ВАШ_КОД>

# Разбиение на train/test уже присутствует в датасете
# структура: dataset['train'], dataset['validation'], dataset['test']
train_data = dataset['train']
val_data = dataset['validation']
test_data = dataset['test']

print(f"Размер обучающей выборки: {len(train_data)}")
print(f"Размер валидационной выборки: {len(val_data)}")
print(f"Размер тестовой выборки: {len(test_data)}")

# TODO 2.2: Загрузите модель FinBERT с HuggingFace.
# Подсказка: используйте AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
# и AutoTokenizer.from_pretrained("ProsusAI/finbert")
model_name = "ProsusAI/finbert"
tokenizer = # <ВАШ_КОД>
model = # <ВАШ_КОД>

# TODO 2.3: Создайте pipeline для анализа тональности.
# Подсказка: используйте pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)
sentiment_pipeline = # <ВАШ_КОД>

# ================================
# 3. ЗАГРУЗКА И ПОДГОТОВКА РЫНОЧНЫХ ДАННЫХ
# ================================
print("\n" + "=" * 60)
print("ЗАГРУЗКА РЫНОЧНЫХ ДАННЫХ")
print("=" * 60)

# TODO 3.1: Загрузите исторические данные индекса S&P 500 за последние 5 лет.
# Используйте yfinance, тикер "^GSPC"
ticker = "^GSPC"
market_data = # <ВАШ_КОД>

# TODO 3.2: Рассчитайте ежедневную доходность и волатильность (стандартное отклонение доходности).
# Доходность: логарифмическая или простая
# Волатильность: скользящее стандартное отклонение (rolling window = 22 дня)
market_returns = # <ВАШ_КОД>
market_volatility = # <ВАШ_КОД>

# ================================
# 4. АНАЛИЗ ТОНАЛЬНОСТИ С ПОМОЩЬЮ FINBERT
# ================================
print("\n" + "=" * 60)
print("АНАЛИЗ ТОНАЛЬНОСТИ FINBERT")
print("=" * 60)

# Функция для анализа тональности с датафрейма
def analyze_sentiment(df, text_column='sentence'):
    """
    Применяет FinBERT ко всем текстам в датафрейме.
    """
    # TODO 4.1: Примените sentiment_pipeline ко всем текстам в колонке text_column.
    # Результат должен содержать метку (label) и уверенность (score).
    # Подсказка: используйте .apply() и lambda функцию
    predictions = # <ВАШ_КОД>
    return predictions

# Применение к тестовой выборке
test_df = test_data.to_pandas()
test_sentiment = analyze_sentiment(test_df)

# TODO 4.2: Оцените качество модели на тестовой выборке.
# Сравните предсказанные метки с истинными (test_data['label']).
# Выведите classification_report и confusion_matrix.
print("\nРезультаты классификации на тестовой выборке:")
# <ВАШ_КОД>

# ================================
# 5. ВРЕМЕННАЯ АГРЕГАЦИЯ СЕНТИМЕНТА
# ================================
print("\n" + "=" * 60)
print("АГРЕГАЦИЯ СЕНТИМЕНТА")
print("=" * 60)

# TODO 5.1: Поскольку в датасете нет дат, создайте синтетический временной ряд.
# Распределите тексты по дням (например, 10-20 текстов в день).
np.random.seed(42)
dates = pd.date_range(start='2020-01-01', periods=len(test_df), freq='D')
test_df['date'] = dates

# TODO 5.2: Сгруппируйте тексты по дням и агрегируйте тональность.
# Для каждого дня вычислите:
# - средний score (neutral_score, positive_score, negative_score)
# - среднюю уверенность
# - количество текстов
daily_sentiment = # <ВАШ_КОД>

# TODO 5.3: Постройте индекс сентимента (Sentiment Index).
# Формулы на выбор:
# SI = (pos_avg - neg_avg) / (pos_avg + neg_avg + neutral_avg)
# Или SI = pos_avg - neg_avg
sentiment_index = # <ВАШ_КОД>

# ================================
# 6. ОБЪЕДИНЕНИЕ С РЫНОЧНЫМИ ДАННЫМИ
# ================================
print("\n" + "=" * 60)
print("ОБЪЕДИНЕНИЕ ДАННЫХ")
print("=" * 60)

# TODO 6.1: Объедините daily_sentiment с market_data по датам.
merged_data = # <ВАШ_КОД>

# TODO 6.2: Создайте лагированные признаки тональности (lag 1, 2, 3, 5 дней)
for lag in [1, 2, 3, 5]:
    merged_data[f'sentiment_lag_{lag}'] = # <ВАШ_КОД>

# ================================
# 7. КРОСС-КОРРЕЛЯЦИОННЫЙ АНАЛИЗ
# ================================
print("\n" + "=" * 60)
print("КРОСС-КОРРЕЛЯЦИОННЫЙ АНАЛИЗ")
print("=" * 60)

# TODO 7.1: Рассчитайте корреляцию Пирсона между индексом сентимента и доходностью/волатильностью.
corr_pearson = merged_data[['sentiment_index', 'returns', 'volatility']].corr()

# TODO 7.2: Рассчитайте ранговую корреляцию Спирмена.
corr_spearman = # <ВАШ_КОД>

# TODO 7.3: Вычислите кросс-корреляцию между тональностью и доходностью (lag = -10..10).
nlags = 10
cross_corr = {}
for lag in range(-nlags, nlags+1):
    if lag < 0:
        # sentiment опережает рынок
        cross_corr[lag] = merged_data['sentiment_index'].corr(merged_data['returns'].shift(-lag))
    else:
        # рынок опережает sentiment
        cross_corr[lag] = merged_data['sentiment_index'].shift(lag).corr(merged_data['returns'])

# ================================
# 8. ВИЗУАЛИЗАЦИЯ
# ================================
print("\n" + "=" * 60)
print("ВИЗУАЛИЗАЦИЯ")
print("=" * 60)

# TODO 8.1: Постройте график временных рядов: цена, индекс сентимента, волатильность
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)
# <ВАШ_КОД>

# TODO 8.2: Постройте тепловую карту корреляций
plt.figure(figsize=(10, 8))
sns.heatmap(corr_pearson, annot=True, cmap='coolwarm', center=0)
plt.title("Корреляции между тональностью и рыночными метриками")
# <ВАШ_КОД>

# TODO 8.3: Постройте график кросс-корреляции
plt.figure(figsize=(12, 5))
lags = list(cross_corr.keys())
values = list(cross_corr.values())
plt.bar(lags, values, color='steelblue', alpha=0.7)
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
plt.axhline(y=0.2, color='red', linestyle='--', alpha=0.7)
plt.axhline(y=-0.2, color='red', linestyle='--', alpha=0.7)
plt.title("Кросс-корреляция: Индекс сентимента vs Доходность")
plt.xlabel("Лаг (отрицательный = сентимент опережает рынок)")
plt.ylabel("Корреляция")
plt.grid(alpha=0.3)
# <ВАШ_КОД>

# ================================
# 9. СТАТИСТИЧЕСКОЕ ТЕСТИРОВАНИЕ ГИПОТЕЗ
# ================================
print("\n" + "=" * 60)
print("СТАТИСТИЧЕСКОЕ ТЕСТИРОВАНИЕ")
print("=" * 60)

# TODO 9.1: Проверьте, является ли корреляция статистически значимой.
# Используйте scipy.stats.pearsonr для расчета p-value.
p_values = {}
for col in ['returns', 'volatility']:
    corr, p_value = # <ВАШ_КОД>
    p_values[col] = p_value
    print(f"Корреляция sentiment_index vs {col}: r={corr:.3f}, p-value={p_value:.5f}")

# TODO 9.2: Проверьте гипотезу о наличии опережающей способности тональности.
# Сравните корреляцию на лагах 0, -1, -2.
best_lag = # <ВАШ_КОД>
print(f"Максимальная кросс-корреляция достигнута при лаге {best_lag}")


ModuleNotFoundError: No module named 'datasets'